Don't use this, i.e. deltakit. Just use deltakit_stim (see leakage_intro_to_deltakit_stim). Deltakit requires simulating on a Riverlane server rather than locally. It seems less functional and clashes more when importing stuff.

In [ ]:
## DELTAKIT STIM ##
import deltakit_stim
# Deltakit circuit clashes with deltakit_stim unless do these injections
injections = ['stim._detect_machine_architecture', 'stim._stim_polyfill', 'stim']
import sys
for namespace in injections:
    sys.modules[namespace] = sys.modules[f"deltakit_{namespace}"]

## DELTAKIT ##
import deltakit.circuit # https://github.com/Deltakit/deltakit/blob/80087d51bbdbdafc149d1eb3882d53febbcf2216/deltakit-circuit/src/deltakit_circuit/_circuit.py
import deltakit.explorer
import deltakit.decode

## DELTAKIT CLIENT ## 
import os
os.environ["DELTAKIT_TOKEN"] = "-gacmW8zIhylEigvAtdul_hTBDNevfVF"
cloud = deltakit.explorer.Client.get_instance()

## BB_IONS ##
sys.path.append(os.path.abspath("../src"))
from bb_ions import *

In [ ]:
# Converting a deltakit_stim circuit to a deltakit_circuit circuit

circuit_string = """
R 0 1
LEAKAGE(1) 0
CX 0 1
TICK
HERALD_LEAKAGE_EVENT() 0 1
M 0 1
DETECTOR rec[-2]
DETECTOR rec[-1]
"""

stim_circuit = deltakit_stim.Circuit(circuit_string) # need to use deltakit_stim because circuit has LEAKAGE etc. 
# Note any reference to 'stim_circuit' below is actually a deltakit_stim circuit which can have LEAKAGE etc. gates.
print(type(stim_circuit))

# This is NOT a deltakit.circuit.Circuit, it must be converted to it:
dk_circuit = deltakit.circuit.Circuit.from_stim_circuit(stim_circuit)

print(type(dk_circuit)) # Success!!! Turns out zipping the repo and chucking it into gippity was the way to go.

In [ ]:
# Trying this on one of my stim circuits:

code = gross_code()
p = 0.001
memory_basis = 'X'

stim_circuit = make_BB_circuit(  # see src/bb_ions/circfuncs for explanation of make_BB_circuit inputs
    code,
    p,  
    errors = helios_errors(p),
    idle_during = helios_idle_errors(p),
    num_syndrome_extraction_cycles = code.d_max,  
    memory_basis = memory_basis,
    sequential_gates = True,
    exclude_opposite_basis_detectors = True,
    reuse_check_qubits = True,
    swap_LRC = False,
    only_CZs = True
)

# print(type(stim_circuit))
# svg = stim_circuit.without_noise().diagram('timeline-svg')
# display(svg)
# with open(f"../scrap.svg", "w", encoding="utf-8") as f: f.write(str(svg))

In [ ]:
## Replace noise and add leakage using one of their noise functions:

# si1000_leakage_noise = deltakit.explorer.qpu.SI1000Noise(p=1e-3, pL=1e-3)
# stim_text = cloud.add_noise(stim_circuit, si1000_leakage_noise)

In [ ]:
# # Update stim circuit as the noisy one by converting the text to a stim_circuit:
# stim_circuit = deltakit_stim.Circuit(stim_text)

## Visualise:
# svg = stim_circuit.diagram('timeline-svg'); display(svg)
# with open(f"../scrap2.svg", "w", encoding="utf-8") as f: f.write(str(svg)) # Adding leakage and noise with their function has changed the qubit indices but not the coordinates so timeslice still looks good (albeit divided into extra timesteps)

In [ ]:
# Simulate with detectors:

# Simulate then convert measurements to detector results: (https://deltakit-docs.riverlane.com/en/stable/guide/decoding.html)
# (Without 'sweep' file assumes circuit starts all qubits in |0⟩ )

measurements = stim_circuit.compile_sampler().sample(10)

In [ ]:
deltakit_measurements = deltakit.explorer.types.Measurements(measurements)
detectors, observables = deltakit_measurements.to_detectors_and_observables(stim_circuit)

print("Measurements:", measurements.shape) # shape returns (num_rows = num_samples, num_columns = num_meas)
print("Detectors   :", detectors.as_numpy().shape)
print("Observables :", observables.as_numpy().shape)

In [ ]:
# Decoding! 

# Convert to deltakit_circuit_circuit
dk_circuit = deltakit.circuit.Circuit.from_stim_circuit(stim_circuit)
print(type(dk_circuit)) # Success !!

In [ ]:
## Decoding from detectors (https://deltakit-docs.riverlane.com/en/stable/guide/decoding.html)

decoder = deltakit.decode.BPOSDecoder(  # https://deltakit-docs.riverlane.com/en/stable/_build/generated/deltakit.decode.BPOSDecoder.html#deltakit.decode.BPOSDecoder
    circuit=dk_circuit,
    parameters={
        "max_bp_rounds": 10_000,
        "combination_sweep_order": 5
        # Can't specify if it's doing minium_sum or product sum ?
    },
    client=cloud
    )


print(f"{decoder.__class__.__name__}: ", end="")
predictions = decoder.decode_batch_to_logical_flip(detectors.as_numpy())

mismatch = (predictions != observables.as_numpy())
fails = int(sum(np.any(mismatch, axis=1)))
print(f"{deltakit.explorer.types.DecodingResult(fails, predictions.shape[0])}")